# Exercice 2 — DataPipeline "Pro"

**Objectif :** Concevoir un pipeline robuste capable de transformer des données brutes en informations décisionnelles.


- Créez une classe nommée DataPipeline comprenant les méthodes suivantes :

1. __init__(self, filepath) : Reçoit le chemin du fichier CSV et charge les données dans un attribut self.df.

2. clean(self) :

    - Supprime les doublons exacts.
    - Convertit toutes les colonnes de type "object" (texte) en minuscules.
    - Supprime les espaces inutiles en début et fin de chaîne (strip).

3. handle_missing_values(self, strategy="mean") :

    - Si strategy="mean", remplace les valeurs numériques NaN par la moyenne de leur colonne.
    - Si strategy="drop", supprime les lignes contenant des valeurs manquantes.

4. calculate_revenue(self) : Crée une nouvelle colonne Revenu_Net en calculant : Ventes * (1 - Remise).

5. get_city_stats(self) : Retourne un groupement par Ville affichant la somme du Revenu_Net et la moyenne des Ventes.

6. filter_by_value(self, colonne, seuil) : Retourne un nouveau DataFrame contenant uniquement les lignes où la colonne est supérieure au seuil.

7. save_report(self, filename) : Génère les statistiques descriptives (.describe()) du DataFrame actuel et les sauvegarde dans un fichier CSV.

8. export_to_json(self, filename) : Sauvegarde le DataFrame final au format JSON (orienté records).

1. Importer la bibliothèque

In [8]:
import pandas as pd

2. Définir la classe DataPipeline

In [9]:
class DataPipeline:

    def __init__(self, filepath):
        self.filepath = filepath
        self.df = pd.read_csv(filepath)

    def clean(self):
        self.df = self.df.drop_duplicates()

        object_cols = self.df.select_dtypes(include="object").columns

        for col in object_cols:
            self.df[col] = (
                self.df[col]
                .str.strip()
                .str.lower()
            )

        return self

    def handle_missing_values(self, strategy="mean"):
        if strategy == "mean":
            numeric_cols = self.df.select_dtypes(include="number").columns

            for col in numeric_cols:
                self.df[col] = self.df[col].fillna(
                    self.df[col].mean()
                )

        elif strategy == "drop":
            self.df = self.df.dropna()

        return self

    def calculate_revenue(self):
        self.df["Revenu_Net"] = (
            self.df["Ventes"] * (1 - self.df["Remise"])
        )

        return self
    
    def get_city_stats(self):
        return (
            self.df.groupby("Ville")
            .agg({
                "Revenu_Net": "sum",
                "Ventes": "mean"
            })
            .reset_index()
        )
    
    def filter_by_value(self, colonne, seuil):
        return self.df[self.df[colonne] > seuil]
    
    def save_report(self, filename):
        self.df.describe().reset_index().to_csv(filename, index=False)

    def export_to_json(self, filename):
        self.df.to_json(
            filename,
            orient="records",
            indent=4
        )

5. Ajouter le calcul du revenu net

In [10]:
pipeline = DataPipeline("dataset.csv")

# Execute the pipeline
pipeline.clean() \
        .handle_missing_values("mean") \
        .calculate_revenue()

C:\Users\Youcode\AppData\Local\Temp\ipykernel_6460\2062882556.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = self.df.select_dtypes(include="object").columns


**Display cleaned DataFrame**

In [11]:
pipeline.df.head()

,ID,Ville,Produit,Ventes,Remise,Revenu_Net
0,101,casablanca,laptop,1250.5,0.10,1125.45
1,102,rabat,mouse,45.0,0.00,45.00
2,103,tanger,screen,300.2,0.15,255.17
3,104,marrakech,keyboard,85.0,0.05,80.75
5,105,fès,tablet,500.0,0.20,400.00


**Display Stats by city**

In [12]:
pipeline.get_city_stats()

,Ville,Revenu_Net,Ventes
0,agadir,459.60,127.50
1,casablanca,3369.95,754.10
2,fès,1748.00,500.50
3,marrakech,1753.55,493.75
4,oujda,866.50,318.50
5,rabat,360.70,95.05
6,tanger,1845.17,492.55


In [13]:
pipeline.filter_by_value("Ventes", 500)

,ID,Ville,Produit,Ventes,Remise,Revenu_Net
0,101,casablanca,laptop,1250.5,0.10,1125.45
8,108,casablanca,smartphone,890.0,0.10,801.00
12,112,fès,laptop,1300.0,0.10,1170.00
16,116,tanger,laptop,1150.0,0.05,1092.50
20,120,oujda,smartphone,850.0,0.10,765.00
24,124,marrakech,laptop,1200.0,0.08,1104.00
29,128,casablanca,smartphone,910.0,0.15,773.50


In [14]:
pipeline.save_report("rapport_stats.csv")
pipeline.export_to_json("dataset_final.json")